# 02 - Metric Anomaly Detection

Notebook này dùng hai hướng anomaly detection:

1. Rolling Z-score cho từng metric theo thời gian.
2. Isolation Forest cho anomaly đa biến trên `cart-service`.

Rolling Z-score hữu ích để thấy metric nào lệch khỏi baseline. Isolation Forest hữu ích hơn khi incident là tổ hợp nhiều tín hiệu: memory, GC, latency, 5xx và restart.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

def find_project_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "g2-data" / "g2").exists():
            return candidate
    raise FileNotFoundError("Cannot find project root containing g2-data/g2")

ROOT = find_project_root()
DATA = ROOT / "g2-data" / "g2"
METRICS = DATA / "metrics"
PLOTS = ROOT / "lab" / "plots"
RESULTS = ROOT / "lab" / "results"
PLOTS.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

def load_metric(name):
    df = pd.read_csv(METRICS / f"{name}.csv")
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    return df

cart = load_metric("cart-service")
cart["memory_pct"] = 100 * cart["memory_usage_bytes"] / cart["memory_limit_bytes"]
cart["restart_delta"] = cart["container_restart_count"].diff().fillna(0)

## 2.1 Rolling Z-score

In [ ]:
def rolling_z(series, window=120, min_periods=60):
    baseline = series.shift(1).rolling(window=window, min_periods=min_periods)
    return (series - baseline.mean()) / baseline.std()

z_rows = []
for metric in ["memory_usage_bytes", "jvm_gc_pause_ms_avg", "http_p99_latency_ms"]:
    z = rolling_z(cart[metric].astype(float))
    hits = cart[z >= 3]
    if not hits.empty:
        idx = hits.index[0]
        z_rows.append({
            "metric": metric,
            "first_z_anomaly": cart.loc[idx, "timestamp"],
            "z_score": z.loc[idx],
            "value": cart.loc[idx, metric],
        })
pd.DataFrame(z_rows)

Raw Rolling Z-score có thể bắt outlier nhỏ từ sớm. Vì vậy, khi viết postmortem, chúng ta không chỉ dựa vào một điểm z-score, mà dùng thêm sustained threshold để xác nhận tín hiệu có ý nghĩa vận hành.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
for ax, metric in zip(axes, ["memory_pct", "jvm_gc_pause_ms_avg", "http_p99_latency_ms"]):
    z = rolling_z(cart[metric].astype(float))
    ax.plot(cart["timestamp"], z, linewidth=1)
    ax.axhline(3, color="#d62728", linestyle="--", linewidth=1)
    ax.set_ylabel(metric)
    ax.grid(True, alpha=0.25)
fig.suptitle("Rolling Z-score trên cart-service")
fig.tight_layout()
fig.savefig(PLOTS / "02_metric_rolling_z_cart.png", dpi=160)
plt.show()

**Rolling Z-score trên cart-service**

![Rolling Z-score trên cart-service](../plots/02_metric_rolling_z_cart.png)

## 2.2 Sustained threshold để xác nhận mốc vận hành

In [ ]:
def first_sustained(df, series, threshold, points=5):
    mask = series >= threshold
    sustained = mask.rolling(points).sum() >= points
    if not sustained.any():
        return None
    idx = np.where(sustained)[0][0] - points + 1
    return df.loc[idx, "timestamp"], series.iloc[idx]

checks = [
    ("cart-service", "jvm_gc_pause_ms_avg", cart["jvm_gc_pause_ms_avg"], 100),
    ("cart-service", "memory_pct", cart["memory_pct"], 70),
    ("cart-service", "http_5xx_rate", cart["http_5xx_rate"], 5),
]
rows = []
for service, metric, series, threshold in checks:
    result = first_sustained(cart, series, threshold)
    if result:
        ts, value = result
        rows.append({"service": service, "metric": metric, "first_sustained": ts, "threshold": threshold, "value": value})
pd.DataFrame(rows)

## 2.3 Isolation Forest trên cart-service

In [ ]:
feature_cols = ["memory_pct", "jvm_gc_pause_ms_avg", "http_p99_latency_ms", "http_5xx_rate", "restart_delta"]
features = cart[feature_cols].astype(float).ffill().bfill()
scaled = StandardScaler().fit_transform(features)

model = IsolationForest(n_estimators=300, contamination=0.04, random_state=42)
pred = model.fit_predict(scaled)
score = -model.decision_function(scaled)
cart["if_anomaly"] = pred == -1
cart["if_score"] = score

first_idx = cart.index[cart["if_anomaly"]][0]
cart.loc[first_idx, ["timestamp"] + feature_cols + ["if_score"]]

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(cart["timestamp"], cart["memory_pct"], label="memory %", color="#1f77b4")
ax2 = ax.twinx()
ax2.plot(cart["timestamp"], cart["jvm_gc_pause_ms_avg"], label="GC pause", color="#9467bd", alpha=0.75)
ax.axvline(cart.loc[first_idx, "timestamp"], color="#111111", linestyle="--", label="Isolation Forest first anomaly")
ax.axhline(70, color="#d62728", linestyle=":", linewidth=1)
ax.set_title("Isolation Forest anomaly đặt trên memory và GC")
ax.set_ylabel("memory %")
ax2.set_ylabel("GC pause ms")
ax.grid(True, alpha=0.25)
lines, labels = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines + lines2, labels + labels2, fontsize=8, loc="upper left")
fig.tight_layout()
fig.savefig(PLOTS / "02_metric_isolation_forest_cart.png", dpi=160)
plt.show()

**Isolation Forest trên cart-service**

![Isolation Forest trên cart-service](../plots/02_metric_isolation_forest_cart.png)

Isolation Forest bắt anomaly đầu tiên ở khoảng `18:04:30Z`, rất gần với mốc GC sustained anomaly `18:06Z`. Đây là evidence mạnh rằng `cart-service` đã bước vào trạng thái bất thường trước khi OOM và restart xảy ra.